### 防御性+收益补偿回测

在策略的回测基础上加入了策略校验逻辑

In [3]:
# -*- coding: utf-8 -*-
"""固定 SVM 因子的防御性市值分组回测：直接粘贴到 BigQuant Notebook 一个单元运行。"""



import importlib.util
import sys
import time
from pathlib import Path

import pandas as pd
import matplotlib as mpl
from IPython.display import display

# 清除本 Notebook 之前单元遗留的不存在中文字体配置，
# 避免 BigQuant 后续渲染时反复输出 findfont 警告。
mpl.rcdefaults()


# ============================== 回测参数 ==============================
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", "/home/aiuser/work"))
MODEL_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "factor_lib"
    / "model_artifacts"
    / "svm_model_bundles"
    / "svm_all_a_fixed_20260831_223116_5bfc17b8"
)

# 模型训练的信息截止日是 2025-02-14；严格样本外回测从其后开始。
BACKTEST_START_DATE = "2023-02-18"
BACKTEST_END_DATE = "2026-08-28"
REBALANCE_INTERVAL = 120

# 五等分市值，仅使用第 1 组（最小市值组）；在该组内买入 svm_score 最高的 5%。
MARKET_CAP_GROUP_COUNT = 5
SELECTED_MARKET_CAP_GROUPS = [1]
FACTOR_QUANTILE_RANGE = (0.95, 1.00)

# 回测表现基准与防御触发基准均为中证 2000（932000）。
BACKTEST_BENCHMARK = "932000.CSI"
DEFENSIVE_BENCHMARK_INDEX = "csi_2000"
DEFENSIVE_MA_WINDOW = 30
# 中证 2000 的信号日收盘价低于 MA60 时，原 SVM 组合仅保留 5% 仓位。
DEFENSIVE_STRATEGY_WEIGHT = 0.05
# 可自行修改；触发防御后其余 95% 按此列表等权配置。
DEFENSIVE_COMPENSATION_INSTRUMENTS = [
    "601398.SH",
    "601328.SH",
    "601988.SH",
]

INITIAL_CASH = 1_000_000
TRADING_COSTS = {
    "buy_cost": 0.0003,
    "sell_cost": 0.0003,
    "min_cost": 5.0,
    "tax_ratio": 0.0005,
}
SLIPPAGE_VALUE = 0.001
VOLUME_LIMIT = 0.025
PROGRESS_EVERY = 1


def _find_project_root(root_hint):
    root_hint = Path(root_hint).expanduser()
    candidates = [root_hint, *root_hint.parents, Path.cwd(), *Path.cwd().parents]
    project_root = next(
        (item.resolve() for item in candidates if (item / "factor_lib").is_dir()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError(
            "未找到项目根目录；BigQuant 环境通常应为 /home/aiuser/work。"
        )
    return project_root


def _load_module(module_name, module_path):
    if not module_path.is_file():
        raise FileNotFoundError(f"找不到脚本：{module_path}")
    module_spec = importlib.util.spec_from_file_location(module_name, module_path)
    if module_spec is None or module_spec.loader is None:
        raise ImportError(f"无法加载脚本：{module_path}")
    module = importlib.util.module_from_spec(module_spec)
    sys.modules[module_name] = module
    module_spec.loader.exec_module(module)
    return module


PROJECT_ROOT = _find_project_root(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if not MODEL_ARTIFACT_DIR.is_dir():
    raise FileNotFoundError(f"找不到模型包目录：{MODEL_ARTIFACT_DIR}")

SVM_MODULE_PATH = (
    PROJECT_ROOT
    / "factor_lib"
    / "Factor Repository"
    / "machine_learning_factors"
    / "svm_score.py"
)
svm_module = _load_module("svm_score_defensive_backtest", SVM_MODULE_PATH)
model_bundle = svm_module.load_svm_model_bundle(MODEL_ARTIFACT_DIR)

from factor_lib.function.bigquant_function.strategies.run_defensive_market_cap_group_backtest import (  # noqa: E402
    run_defensive_market_cap_group_backtest,
)


# 在同一 Notebook 内保留已经推理完成的单日 SVM 截面；调整回测参数时不会
# 重复读取十个依赖特征，且每次推理仍只使用相应信号日及此前的数据。
SVM_SCORE_CACHE_KEY = (str(MODEL_ARTIFACT_DIR.resolve()), "all_a")
_notebook_globals = globals()
_score_panel_cache = _notebook_globals.get("_SVM_SCORE_PANEL_CACHE")
if _score_panel_cache is None:
    _score_panel_cache = {}
    _notebook_globals["_SVM_SCORE_PANEL_CACHE"] = _score_panel_cache
if not isinstance(_score_panel_cache, dict):
    raise TypeError("_SVM_SCORE_PANEL_CACHE 必须是字典；请重启内核后重试。")
_score_cache_by_date = _score_panel_cache.setdefault(
    SVM_SCORE_CACHE_KEY,
    {},
)


def svm_factor_panel_provider(signal_dates):
    """按信号日流式推理 SVM，避免一次性加载全部特征历史导致内存不足。"""
    dates = (
        pd.DatetimeIndex(pd.to_datetime(signal_dates))
        .normalize()
        .unique()
        .sort_values()
    )
    started_at = time.perf_counter()
    missing_dates = [
        signal_date
        for signal_date in dates
        if signal_date not in _score_cache_by_date
    ]
    print(
        f"[防御性 SVM 市值分组回测] 需要 {len(dates)} 个信号日评分；"
        f"缓存命中 {len(dates) - len(missing_dates)} 个，"
        f"待流式推理 {len(missing_dates)} 个。",
        flush=True,
    )
    for position, signal_date in enumerate(missing_dates, start=1):
        print(
            f"[防御性 SVM 市值分组回测] 流式推理 "
            f"{position}/{len(missing_dates)} | 当前 {signal_date:%Y-%m-%d} "
            f"| 已耗时 {time.perf_counter() - started_at:.1f}s",
            flush=True,
        )
        score = svm_module.infer_svm_score(
            target_dates=[signal_date],
            model_bundle=model_bundle,
            universe={"type": "all_a"},
            as_of_date=signal_date,
            batch_signal_dates=1,
            show_progress=True,
            progress_every=1,
        )
        _score_cache_by_date[signal_date] = score.loc[
            :, ["date", "instrument", "svm_score"]
        ].copy()

    score_panel = pd.concat(
        [_score_cache_by_date[signal_date] for signal_date in dates],
        axis=0,
        ignore_index=True,
    )
    score_panel["date"] = pd.to_datetime(score_panel["date"]).dt.normalize()
    return score_panel.drop_duplicates(["date", "instrument"])


defensive_svm_backtest_result = run_defensive_market_cap_group_backtest(
    start_date=BACKTEST_START_DATE,
    end_date=BACKTEST_END_DATE,
    rebalance_interval=REBALANCE_INTERVAL,
    universe={"type": "all_a"},
    factor_name="svm_score",
    market_cap_group_count=MARKET_CAP_GROUP_COUNT,
    selected_market_cap_groups=SELECTED_MARKET_CAP_GROUPS,
    factor_quantile_range=FACTOR_QUANTILE_RANGE,
    factor_params=None,
    factor_panel_provider=svm_factor_panel_provider,
    defensive_benchmark_index=DEFENSIVE_BENCHMARK_INDEX,
    defensive_ma_window=DEFENSIVE_MA_WINDOW,
    defensive_strategy_weight=DEFENSIVE_STRATEGY_WEIGHT,
    defensive_compensation_instruments=DEFENSIVE_COMPENSATION_INSTRUMENTS,
    order_price_field_buy="open",
    order_price_field_sell="open",
    initial_cash=INITIAL_CASH,
    benchmark=BACKTEST_BENCHMARK,
    trading_costs=TRADING_COSTS,
    slippage_value=SLIPPAGE_VALUE,
    volume_limit=VOLUME_LIMIT,
    show_progress=True,
    progress_every=PROGRESS_EVERY,
)

display(pd.DataFrame([defensive_svm_backtest_result["data_diagnostics"]]))
display(defensive_svm_backtest_result["rebalance_audit"].tail(10))

# 新版策略的日频 MA 审计：仅显示发生因子调仓或进入/退出防御的日期。
daily_defensive_audit = defensive_svm_backtest_result["defensive_audit"].copy()
daily_defensive_audit["decision_date"] = pd.to_datetime(
    daily_defensive_audit["decision_date"]
).dt.normalize()
daily_defensive_audit["execution_date"] = pd.to_datetime(
    daily_defensive_audit["execution_date"]
).dt.normalize()
daily_action_audit = daily_defensive_audit.loc[
    daily_defensive_audit["action_required"],
    [
        "decision_date",
        "execution_date",
        "event_type",
        "is_defensive",
        "market_close",
        "market_ma",
        "defense_entry",
        "defense_exit",
        "factor_rebalance_due",
        "main_strategy_target_weight",
        "compensation_target_weight",
        "order_intent_count",
        "submitted_order_count",
        "blocked_order_count",
    ],
].copy()
display(daily_action_audit)


display(defensive_svm_backtest_result["performance"])





[BigQuant 市场日频适配器] [3/3] 市场日频数据准备完成 | 886 行 | 已耗时 0.0s                                                                                                                                                                      
[市值分组回测] [2/8] 调用流式因子面板提供器 | 0/1 (0.0%) | 当前 svm_score | 8个信号日 | 已耗时 0.2s                                                                                                                                                   [防御性 SVM 市值分组回测] 需要 8 个信号日评分；缓存命中 8 个，待流式推理 0 个。


[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 42,877 行 | 已耗时 1.8s                                                                                                                                                        
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 287,789 行 | 已耗时 1.9s                                                                                                                                                       
[市值分组回测] [6/8] 启动 BigTrader：日频 MA 监控与必要调仓 | 0/857 (0.0%) | 因子计划8次 | 已耗时 13.3s                                                                                                                                               
[2026-09-02 13:59:48] [info     ] bigtrader.v35 开始运行 ..
[2026-09-02 13:59:48] [info     ] pybacktest run 2023-02-17 ~ 2026-08-28, , equity, instruments=337
[2026-09-02 13:59:48] [info     ] bigtrader module V2.2.0
[2026-09-02 13:59:48] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10
[2026-09-02 13:59:49] [info     ] pybac

[2026-09-02 14:00:00] [info     ] bigtrader.v35 运行完成 [11.945s].
[市值分组回测] [8/8] 回测与审计结果整理完成 | 1/1 (100.0%) | 因子信号8次，日频审计857日，成交4968条 | 已耗时 25.3s                                                                                                                                             


,requested_start_date,actual_first_execution_date,engine_start_date,end_date,factor_history_days,factor_loading_mode,resolved_factor_params,factor_raw_rows,factor_domain_rows,provided_factor_rows,...,execution_state_rows,engine_instrument_count,defensive_config,daily_defensive_day_count,defensive_entry_count,defensive_exit_count,daily_switch_count,scheduled_rebalance_count,successful_signal_count,total_runtime_seconds
0,2023-02-18,2023-02-20,2023-02-17,2026-08-28,None,factor_panel_provider,{},0,{},42877,...,287789,337,"{'benchmark_index': '932000.CSI', 'ma_window':...",404,43,43,93,8,8,25.277908


,rebalance_number,signal_date,execution_date,status,error_message,candidate_count,eligible_count,actual_group_count,group_populations,selected_counts,target_count
0,1,2023-02-17,2023-02-20,ok,,5092,4955,5,"{1: 991, 2: 991, 3: 991, 4: 991, 5: 991}","{1: 50, 2: 0, 3: 0, 4: 0, 5: 0}",50
1,2,2023-08-14,2023-08-15,ok,,5250,5125,5,"{1: 1025, 2: 1025, 3: 1025, 4: 1025, 5: 1025}","{1: 52, 2: 0, 3: 0, 4: 0, 5: 0}",52
2,3,2024-02-07,2024-02-08,ok,,5349,5234,5,"{1: 1047, 2: 1047, 3: 1046, 4: 1047, 5: 1047}","{1: 53, 2: 0, 3: 0, 4: 0, 5: 0}",53
3,4,2024-08-09,2024-08-12,ok,,5356,5213,5,"{1: 1043, 2: 1042, 3: 1043, 4: 1042, 5: 1043}","{1: 53, 2: 0, 3: 0, 4: 0, 5: 0}",53
4,5,2025-02-13,2025-02-14,ok,,5395,5260,5,"{1: 1052, 2: 1052, 3: 1052, 4: 1052, 5: 1052}","{1: 53, 2: 0, 3: 0, 4: 0, 5: 0}",53
5,6,2025-08-07,2025-08-08,ok,,5420,5239,5,"{1: 1048, 2: 1048, 3: 1047, 4: 1048, 5: 1048}","{1: 53, 2: 0, 3: 0, 4: 0, 5: 0}",53
6,7,2026-02-03,2026-02-04,ok,,5478,5291,5,"{1: 1059, 2: 1058, 3: 1058, 4: 1058, 5: 1058}","{1: 53, 2: 0, 3: 0, 4: 0, 5: 0}",53
7,8,2026-08-05,2026-08-06,ok,,5537,5325,5,"{1: 1065, 2: 1065, 3: 1065, 4: 1065, 5: 1065}","{1: 54, 2: 0, 3: 0, 4: 0, 5: 0}",54


,decision_date,execution_date,event_type,is_defensive,market_close,market_ma,defense_entry,defense_exit,factor_rebalance_due,main_strategy_target_weight,compensation_target_weight,order_intent_count,submitted_order_count,blocked_order_count
0,2023-02-17,2023-02-20,factor_rebalance,False,2505.3849,2439.278313,False,False,True,1.00,0.00,50,50,0
12,2023-03-07,2023-03-08,defense_entry,True,2491.9744,2516.535553,True,False,False,0.05,0.95,53,53,0
31,2023-04-03,2023-04-04,defense_exit,False,2514.1235,2502.062670,False,True,False,1.00,0.00,52,51,1
32,2023-04-04,2023-04-06,defense_entry,True,2482.1870,2499.898553,True,False,False,0.05,0.95,53,53,0
34,2023-04-07,2023-04-10,defense_exit,False,2503.8155,2496.168407,False,True,False,1.00,0.00,53,51,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
812,2026-06-26,2026-06-29,defense_entry,True,3446.9849,3529.921313,True,False,False,0.05,0.95,56,56,0
815,2026-07-01,2026-07-02,defense_exit,False,3545.3819,3509.980523,False,True,False,1.00,0.00,56,55,1
816,2026-07-02,2026-07-03,defense_entry,True,3481.7066,3502.682840,True,False,False,0.05,0.95,56,56,0
840,2026-08-05,2026-08-06,factor_rebalance,True,3000.2908,3097.144113,False,False,True,0.05,0.95,97,77,20


{'raw_perf': dai.DataSource("_df20d82d596842878baa009103047ae1"), 'order_price_field_buy': 'open', 'order_price_field_sell': 'open', 'plot_charts': True, 'start_date': '2023-02-17', 'end_date': '2026-08-28', 'capital_base': 1000000.0, 'instruments': None, 'frequency': '1d', 'benchmark': '932000.CSI', 'product_type': 'equity', 'before_start_days': 0, 'volume_limit': 0.025, 'benchmark_data': None, 'basic_data': None, 'dominant_data': None, 'options_data': None, 'debug': False, 'perf_round_num': 4, 'strategy_name': None, 'run_begin_time': '2026-09-02 13:59:48', 'run_mode': 'backtest', 'run_engine': 'py', 'render': <bound method render of {...}>, 'display': <bound method render of {...}>, 'read_raw_perf': <bound method read_raw_perf of {...}>, 'analyze_pnl_per_trade': <bound method analyze_pnl_per_trade of {...}>, 'analyze_pnl_per_day': <bound method analyze_pnl_per_day of {...}>, 'plot_return_curve_per_stock': <bound method plot_return_curve_per_stock of {...}>, 'plot_curve_return': <boun